# 🚀 README Generator using Colab AI

This notebook generates a comprehensive README.md and README.html for your project using **Google Colab's free AI models** (`google.colab.ai`) - **no API key required**.

## How to Use:
1. Upload your project folder to Google Drive
2. Mount Google Drive (Cell 1)
3. Set your project path (Cell 2)
4. Run all cells!

In [1]:
# @title 📁 Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

KeyboardInterrupt: 

In [ ]:
# @title ⚙️ Step 2: Configuration
# @markdown ### Set your project path:
PROJECT_PATH = "/content/drive/MyDrive/Tradingview recreation"  # @param {type:"string"}

# @markdown ### AI Model to use:
MODEL = "gemini-2.5-flash-lite"  # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash"]

# @markdown ### Max files to summarize (limit to avoid rate limits):
MAX_FILES = 40  # @param {type:"slider", min:10, max:100, step:5}

# @markdown ### Rate limit delay (seconds between API calls):
RATE_LIMIT_DELAY = 0.5  # @param {type:"number"}

# @markdown ### Project info:
PROJECT_NAME = "TradingView Recreation"  # @param {type:"string"}
PROJECT_DESCRIPTION = "Production-grade market analysis and trading platform"  # @param {type:"string"}

print(f"📁 Project: {PROJECT_PATH}")
print(f"🤖 Model: {MODEL}")
print(f"📄 Max files: {MAX_FILES}")

In [ ]:
# @title 🔧 Step 3: Setup & Imports
import os
import json
import time
from pathlib import Path
from datetime import datetime

# Check Colab AI availability
try:
    from google.colab import ai
    print("✅ Colab AI is available!")
    print(f"\n📋 Available models:")
    for model in ai.list_models():
        print(f"   • {model}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure you're running this in Google Colab!")

In [ ]:
# @title 📂 Step 4: File Scanner

IGNORE_PATTERNS = [
    "node_modules", "venv", "__pycache__", ".git", "build", "dist",
    "test-results", "playwright-report", ".pytest_cache", "artifacts",
    "*.pyc", "*.log", "*.lock", "package-lock.json", ".env", "*.db",
]

PRIORITY_FILES = [
    "README.md", "package.json", "requirements.txt", "main.py", "main.tsx",
    "App.tsx", "index.ts", "config.py", "settings.py",
]

def should_ignore(path):
    for pattern in IGNORE_PATTERNS:
        if pattern.startswith("*"):
            if path.endswith(pattern[1:]):
                return True
        elif pattern in path:
            return True
    return False

def scan_project(project_path, max_depth=4):
    """Scan project and return list of important files."""
    files = []
    project_root = Path(project_path)

    for root, dirs, filenames in os.walk(project_root):
        dirs[:] = [d for d in dirs if not should_ignore(d)]

        rel_path = Path(root).relative_to(project_root)
        if len(rel_path.parts) > max_depth:
            continue

        for filename in filenames:
            full_path = Path(root) / filename
            rel_file_path = full_path.relative_to(project_root)

            if should_ignore(str(rel_file_path)):
                continue

            ext = full_path.suffix.lower()
            if ext in ['.py', '.ts', '.tsx', '.js', '.jsx', '.json', '.yml', '.yaml', '.md', '.sh']:
                try:
                    size = full_path.stat().st_size
                    if size < 100000:
                        files.append({
                            "path": str(rel_file_path),
                            "name": filename,
                            "ext": ext,
                            "size": size,
                            "priority": filename in PRIORITY_FILES
                        })
                except:
                    pass

    files.sort(key=lambda x: (not x["priority"], x["ext"], x["path"]))
    return files

# Scan the project
print(f"📁 Scanning {PROJECT_PATH}...")
project_files = scan_project(PROJECT_PATH)
print(f"✅ Found {len(project_files)} relevant files")

# Show file type breakdown
ext_counts = {}
for f in project_files:
    ext_counts[f['ext']] = ext_counts.get(f['ext'], 0) + 1
print("\n📊 File types:")
for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1])[:8]:
    print(f"   {ext}: {count} files")

In [ ]:
# @title 🤖 Step 5: AI Content Generation Functions

def read_file_content(file_path, max_chars=3000):
    """Read file content (truncated)."""
    try:
        full_path = Path(PROJECT_PATH) / file_path
        with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read(max_chars)
            if len(content) == max_chars:
                content += "\n... [truncated]"
            return content
    except:
        return ""

def generate_with_ai(prompt):
    """Generate text using Colab AI."""
    try:
        response = ai.generate_text(prompt, model=MODEL)
        time.sleep(RATE_LIMIT_DELAY)
        return response.strip()
    except Exception as e:
        print(f"   ⚠️ AI Error: {e}")
        return f"[Error: {e}]"

def summarize_file(file_path, content):
    """Generate 1-2 sentence summary of a file."""
    prompt = f"""Summarize this code file in 1-2 sentences. Focus on its purpose.

File: {file_path}

```
{content[:2000]}
```

Summary:"""
    return generate_with_ai(prompt)

print("✅ AI functions ready!")
print(f"🤖 Using model: {MODEL}")

In [ ]:
# @title 🌲 Step 6: Generate Project Tree

def build_tree(project_path, max_depth=2):
    """Build ASCII project tree."""
    tree_lines = [f"└── {Path(project_path).name}/"]
    project_root = Path(project_path)

    def add_items(path, prefix, depth):
        if depth > max_depth:
            return
        items = []
        try:
            for item in sorted(path.iterdir()):
                if not should_ignore(item.name):
                    items.append(item)
        except:
            return

        for i, item in enumerate(items[:12]):
            is_last = i == len(items) - 1 or i == 11
            connector = "└── " if is_last else "├── "

            if item.is_dir():
                tree_lines.append(f"{prefix}{connector}{item.name}/")
                if depth < max_depth:
                    new_prefix = prefix + ("    " if is_last else "│   ")
                    add_items(item, new_prefix, depth + 1)
            else:
                tree_lines.append(f"{prefix}{connector}{item.name}")

        if len(items) > 12:
            tree_lines.append(f"{prefix}└── ... ({len(items) - 12} more)")

    add_items(project_root, "    ", 0)
    return "\n".join(tree_lines)

project_tree = build_tree(PROJECT_PATH)
print("🌲 Project Structure:\n")
print(project_tree)

In [ ]:
# @title 🏷️ Step 7: Detect Tech Stack & Generate Badges

def detect_stack():
    """Detect technologies from project files."""
    stack = {"languages": set(), "frameworks": set(), "tools": set()}

    ext_map = {".py": "Python", ".ts": "TypeScript", ".tsx": "React", ".js": "JavaScript"}
    for f in project_files:
        if f["ext"] in ext_map:
            stack["languages"].add(ext_map[f["ext"]])

    # Check package.json
    for pkg_path in ["frontend/package.json", "package.json"]:
        try:
            with open(Path(PROJECT_PATH) / pkg_path) as f:
                pkg = json.load(f)
                deps = {**pkg.get("dependencies", {}), **pkg.get("devDependencies", {})}
                if "react" in deps: stack["frameworks"].add("React")
                if "vite" in deps: stack["tools"].add("Vite")
                if "tailwindcss" in deps: stack["tools"].add("Tailwind")
                if "zustand" in deps: stack["frameworks"].add("Zustand")
        except: pass

    # Check requirements.txt
    for req_path in ["phase1/requirements.txt", "requirements.txt"]:
        try:
            with open(Path(PROJECT_PATH) / req_path) as f:
                reqs = f.read().lower()
                if "fastapi" in reqs: stack["frameworks"].add("FastAPI")
                if "sqlalchemy" in reqs: stack["frameworks"].add("SQLAlchemy")
                if "pandas" in reqs: stack["tools"].add("Pandas")
        except: pass

    return {k: list(v) for k, v in stack.items()}

def generate_badges(stack):
    """Create shield.io badges."""
    badge_map = {
        "Python": ("Python", "3776AB", "python"),
        "TypeScript": ("TypeScript", "3178C6", "typescript"),
        "React": ("React", "61DAFB", "react"),
        "FastAPI": ("FastAPI", "009688", "fastapi"),
        "Vite": ("Vite", "646CFF", "vite"),
        "Tailwind": ("Tailwind", "06B6D4", "tailwindcss"),
    }
    badges = []
    for category in ["languages", "frameworks", "tools"]:
        for item in stack.get(category, []):
            if item in badge_map:
                name, color, logo = badge_map[item]
                badges.append(f'![{name}](https://img.shields.io/badge/{name}-{color}?style=flat-square&logo={logo}&logoColor=white)')
    return " ".join(badges)

tech_stack = detect_stack()
badges_md = generate_badges(tech_stack)

print("🔍 Detected Tech Stack:")
print(f"   Languages: {', '.join(tech_stack['languages'])}")
print(f"   Frameworks: {', '.join(tech_stack['frameworks'])}")
print(f"   Tools: {', '.join(tech_stack['tools'])}")
print(f"\n🏷️ Badges: {badges_md[:100]}...")

In [ ]:
# @title 📝 Step 8: Generate Overview with AI

print("🤖 Generating project overview...")

file_summary = ", ".join([f"{count} {ext}" for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1])[:5]])
sample_files = "\n".join([f"- {f['path']}" for f in project_files[:15]])

overview_prompt = f"""Write a compelling 2-3 paragraph overview for a README based on this project.

Project: {PROJECT_NAME}
Description: {PROJECT_DESCRIPTION}
Tech Stack: {', '.join(tech_stack['languages'])} + {', '.join(tech_stack['frameworks'])}
Files: {file_summary}

Sample files:
{sample_files}

Write a professional overview explaining what this project does, its key features, and technology stack. Be specific."""

overview_text = generate_with_ai(overview_prompt)
print("\n" + "="*50)
print(overview_text)
print("="*50)

In [ ]:
# @title ✨ Step 9: Generate Features Table with AI

print("🤖 Generating features table...")

features_prompt = f"""Based on these project files, create a markdown features table with 6-8 key features.

Files:
{chr(10).join([f['path'] for f in project_files[:25]])}

Format exactly as:
| Feature | Description |
|---------|-------------|
| Feature Name | Brief description |

Be specific about technical capabilities."""

features_table = generate_with_ai(features_prompt)
print("\n" + features_table)

In [ ]:
# @title 📄 Step 10: Generate File Summaries with AI

print(f"🤖 Generating summaries for {MAX_FILES} files...\n")

file_summaries = []
files_to_process = project_files[:MAX_FILES]

for i, f in enumerate(files_to_process):
    print(f"[{i+1}/{len(files_to_process)}] {f['path'][:60]}...", end=" ")

    content = read_file_content(f["path"])
    if content:
        summary = summarize_file(f["path"], content)
        file_summaries.append({"path": f["path"], "summary": summary})
        print("✓")
    else:
        print("(skipped)")

print(f"\n✅ Generated {len(file_summaries)} file summaries!")

In [ ]:
# @title 📋 Step 11: Build Complete README

print("📝 Building README.md...")

# Build file summaries table
summaries_table = "| File | Description |\n|------|-------------|\n"
for fs in file_summaries[:30]:
    clean_summary = fs['summary'].replace('|', '-').replace('\n', ' ')[:120]
    summaries_table += f"| `{fs['path']}` | {clean_summary} |\n"

readme_content = f"""<div align="center">

# 🚀 {PROJECT_NAME}

**{PROJECT_DESCRIPTION}**

{badges_md}

</div>

---

## 📖 Overview

{overview_text}

---

## ✨ Features

{features_table}

---

## 🏗️ Project Structure

```
{project_tree}
```

---

## 📦 Key Components

{summaries_table}

---

## 🚀 Quick Start

### Prerequisites

- **Python 3.10+** (backend)
- **Node.js 18+** (frontend)
- **Docker** (optional)

### Installation

```bash
# Clone the repository
git clone https://github.com/yourusername/{PROJECT_NAME.lower().replace(' ', '-')}.git
cd {PROJECT_NAME.lower().replace(' ', '-')}

# Backend setup
cd phase1
python -m venv venv
source venv/bin/activate
pip install -r requirements.txt

# Frontend setup
cd ../frontend
npm install
```

### Running

```bash
# Terminal 1 - Backend
cd phase1
uvicorn services.api.main:app --reload --port 8000

# Terminal 2 - Frontend
cd frontend
npm run dev
```

Open [http://localhost:5173](http://localhost:5173) in your browser.

---

## 🧪 Testing

```bash
# Backend tests
cd phase1 && pytest

# Frontend tests
cd frontend && npm test

# E2E tests
npm run test:e2e
```

---

## 📄 License

This project is licensed under the MIT License.

---

<div align="center">

**Generated on {datetime.now().strftime('%Y-%m-%d %H:%M')} using Google Colab AI**

</div>
"""

print(f"✅ README content ready! ({len(readme_content)} characters)")

In [ ]:
# @title 💾 Step 12: Save README.md

output_md_path = f"{PROJECT_PATH}/README_GENERATED.md"

with open(output_md_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"✅ README saved to: {output_md_path}")

In [ ]:
# @title 🌐 Step 13: Generate HTML Version

!pip install markdown -q
import markdown

md = markdown.Markdown(extensions=['extra', 'tables', 'toc'])
html_body = md.convert(readme_content)

html_doc = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{PROJECT_NAME}</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{
            font-family: 'Inter', -apple-system, sans-serif;
            line-height: 1.7;
            color: #e2e8f0;
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
            min-height: 100vh;
            padding: 40px 20px;
        }}
        .container {{ max-width: 900px; margin: 0 auto; }}
        h1 {{ font-size: 2.5rem; margin-bottom: 1rem; color: #f1f5f9; }}
        h2 {{ font-size: 1.75rem; margin: 2rem 0 1rem; color: #818cf8; border-bottom: 2px solid #334155; padding-bottom: 0.5rem; }}
        h3 {{ font-size: 1.25rem; margin: 1.5rem 0 0.75rem; color: #94a3b8; }}
        p {{ margin: 1rem 0; color: #cbd5e1; }}
        a {{ color: #818cf8; text-decoration: none; }}
        a:hover {{ text-decoration: underline; }}
        code {{ background: #1e293b; padding: 2px 6px; border-radius: 4px; font-size: 0.9em; }}
        pre {{ background: #0f172a; padding: 1rem; border-radius: 8px; overflow-x: auto; margin: 1rem 0; border: 1px solid #334155; }}
        pre code {{ background: none; padding: 0; }}
        table {{ width: 100%; border-collapse: collapse; margin: 1rem 0; }}
        th, td {{ padding: 0.75rem; text-align: left; border-bottom: 1px solid #334155; }}
        th {{ background: #1e293b; color: #f1f5f9; }}
        tr:hover {{ background: #1e293b; }}
        img {{ max-height: 28px; margin: 4px; vertical-align: middle; }}
        hr {{ border: none; border-top: 1px solid #334155; margin: 2rem 0; }}
        ul, ol {{ margin: 1rem 0; padding-left: 1.5rem; }}
        li {{ margin: 0.5rem 0; }}
    </style>
</head>
<body>
    <div class="container">
        {html_body}
    </div>
</body>
</html>"""

output_html_path = f"{PROJECT_PATH}/README_GENERATED.html"

with open(output_html_path, 'w', encoding='utf-8') as f:
    f.write(html_doc)

print(f"✅ HTML saved to: {output_html_path}")

In [ ]:
# @title 🎉 Done! Preview the README

from IPython.display import display, Markdown, HTML

print("="*60)
print("🎉 README Generation Complete!")
print("="*60)
print(f"\n📄 Markdown: {output_md_path}")
print(f"🌐 HTML: {output_html_path}")
print(f"\n📊 Stats:")
print(f"   • Files scanned: {len(project_files)}")
print(f"   • Files summarized: {len(file_summaries)}")
print(f"   • README size: {len(readme_content):,} characters")

print("\n" + "="*60)
print("Preview (first 2000 chars):")
print("="*60)
display(Markdown(readme_content[:2000] + "\n\n... [truncated]"))